# 제7-1장. AI 기반 환경 스캐닝

## 학습 목표
1. 전통적 환경분석 프레임워크(PEST, SWOT, 5 Forces)의 역할과 한계를 설명할 수 있다
2. AI 기반 트렌드 탐지와 약신호(Weak Signal) 포착의 원리를 이해한다
3. 뉴스 분석, 감성 분석, 이상 탐지를 코드로 구현할 수 있다
4. 경쟁 인텔리전스(특허, 채용, 제품 분석)를 자동화하는 방법을 학습한다
5. 데이터 기반 SWOT을 작성하고 기회/위협을 정량화할 수 있다

---

### 강의 구조 (3시간)

| 시간 | 구분 | 내용 |
|------|------|------|
| | Part 1 | 환경분석의 목적과 전통적 프레임워크의 한계 |
| | Part 2 | AI 기반 트렌드 탐지: 약신호, 감성 분석, LLM |
| | 휴식 | |
| | Part 3 | 경쟁 인텔리전스: 특허, 채용, 제품 분석 |
| | Part 4 | 실습: 뉴스 트렌드 분석 & 약신호 탐지 |
| | Part 5 | 실습: 경쟁사 프로파일링 & 데이터 기반 SWOT |

---

In [ ]:
# ============================================================
# 환경설정 및 라이브러리 로드
# ============================================================
# 처음 사용 시 아래 명령을 터미널에서 먼저 실행하세요:
#   python setup_env.py
# 그 후 VSCode 우측 상단에서 커널을 'AI 기획 강의 (Python 3)'으로 선택하세요.
# ============================================================

import sys

# 패키지 확인
_required = ["numpy", "pandas", "plotly"]
_missing = []
for _pkg in _required:
    try:
        __import__(_pkg)
    except ImportError:
        _missing.append(_pkg)
if _missing:
    print(f"❌ 누락된 패키지: {', '.join(_missing)}")
    print("   터미널에서 다음 명령을 실행하세요: python setup_env.py")
    raise SystemExit(1)

# 라이브러리 임포트
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("✅ 라이브러리 로드 완료!")

---

## Part 1. 환경분석의 목적과 전통적 프레임워크의 한계

---

### 7.1 환경분석이 왜 필요한가?

환경분석(Environmental Analysis)은 조직 외부의 거시환경, 산업환경, 경쟁환경을 체계적으로 파악하는 활동이다.

**환경분석의 세 가지 목적:**

| 목적 | 핵심 질문 | 산출물 |
|------|----------|--------|
| 기회와 위협 식별 | 환경 변화가 유리한가, 불리한가? | 기회/위협 목록 |
| 전략적 의사결정 지원 | 시장에 진입할 것인가? 투자할 것인가? | 의사결정 근거 |
| 미래 변화 대비 | 환경이 어떻게 변할 것인가? | 시나리오, 조기경보 |

### 전통적 프레임워크

| 프레임워크 | 분석 대상 | 핵심 용도 | 주요 한계 |
|-----------|----------|----------|----------|
| PEST/PESTEL | 거시환경 | 외부 요인 체크 | 체크리스트 수준, 연결 부재 |
| Porter's 5 Forces | 산업환경 | 경쟁 구조 파악 | 산업 경계 모호화 |
| SWOT | 내외부 종합 | 현황 정리 | 주관적, 정성적 |
| Value Chain | 내부 활동 | 원가/차별화 분석 | 복잡한 가치망 반영 어려움 |

이들은 여전히 유용하지만, **네 가지 근본적 한계**가 있다:
1. **정적 분석**: 스냅샷일 뿐, 실시간 변화를 반영하지 못한다
2. **주관성**: 같은 현상에 대해 분석자마다 다른 결론
3. **정보 처리 한계**: 매일 수만 건의 뉴스, 논문, 특허를 인간이 모두 검토 불가
4. **정량화 어려움**: "경쟁 심화"라고 적었는데, 얼마나?

In [ ]:
# 시각화: 전통적 vs AI 기반 환경분석 비교
dimensions = ['Update Frequency', 'Data Coverage', 'Objectivity',
              'Quantification', 'Weak Signal Detection']

traditional = [1, 2, 2, 1, 1]
ai_based = [5, 5, 4, 5, 4]

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=traditional, theta=dimensions, fill='toself',
    name='Traditional (PEST, SWOT)',
    line=dict(color='#95a5a6')
))
fig.add_trace(go.Scatterpolar(
    r=ai_based, theta=dimensions, fill='toself',
    name='AI-Based Scanning',
    line=dict(color='#3498db')
))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 6])),
    title='Traditional vs AI-Based Environmental Analysis',
    height=450
)
fig.show()

print("💡 핵심 차이:")
print("   - 전통적 접근: 연간/분기 → AI 기반: 실시간/일간")
print("   - 전통적 접근: 전문가 판단 → AI 기반: 데이터 + 전문가")
print("   - 전통적 접근: 정성적 서술 → AI 기반: 정량적 지표")
print("\n📌 AI는 전문가를 대체하는 것이 아니라, Human-AI 협업이 핵심이다.")

### 7.1.4 패러다임 전환: 정적 → 동적, 주관 → 데이터

| 구분 | 전통적 접근 | AI 기반 접근 |
|------|-----------|-------------|
| 분석 주기 | 연간/분기 | 실시간/일간 |
| 데이터 범위 | 샘플, 선별 | 대규모, 전수 |
| 분석 방식 | 전문가 판단 | 데이터 기반 + 전문가 |
| 결과 형태 | 정성적 서술 | 정량적 지표 |
| 편향 관리 | 제한적 | 체계적 교차검증 |
| 약신호 탐지 | 어려움 | 자동화 가능 |

> **코로나19 교훈**: 2020년 1월에 수립된 연간 전략은 3월이 되자 무용지물이 되었다.
> 정적 분석의 한계를 극명하게 보여준 사례이다.

---

### 📝 이론 과제 7-1-1 (15분)

#### 과제: 전통적 프레임워크의 한계 사례

**질문:**

1. PEST 분석이 **요인 간 연결(상호작용)**을 보여주지 못하는 한계를 구체적 사례로 설명하시오. 예를 들어, 정치적 요인이 경제적 요인에 어떻게 영향을 미치는지, 그리고 왜 이를 별도로 분석하면 놓칠 수 있는지 설명하시오. (3-4문장)

2. **Porter's 5 Forces가 디지털 시대에 적용하기 어려운 산업** 사례를 하나 검색하시오. 산업 경계가 모호한 이유와 대체재 정의의 어려움을 설명하시오. (3-4문장)

3. Igor Ansoff의 **약신호(Weak Signal)** 개념(1975)을 검색하고, 왜 전통적 환경분석이 약신호를 놓치기 쉬운지 설명하시오. (2-3문장)

**조사 키워드:**
- "PEST analysis limitations interconnections"
- "Porter five forces digital platform"
- "Ansoff weak signal 1975"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-1-1 제출란

**학번:** ___________
**이름:** ___________

#### 1. PEST 분석의 상호작용 한계

(여기에 작성)

#### 2. 5 Forces의 디지털 시대 한계

(여기에 작성)

#### 3. Ansoff의 약신호 개념

(여기에 작성)

**출처:** (URL)

---

## Part 2. AI 기반 트렌드 탐지

---

### 7.2 약신호(Weak Signal) 포착

**약신호(Weak Signal)**는 미래 변화의 초기 징후이다. Igor Ansoff(1975)가 처음 제안하였다.

| 구분 | 약신호 | 강신호 |
|------|--------|--------|
| 명확성 | 불명확, 해석 여지 | 명확, 분명한 의미 |
| 확산도 | 소수만 인지 | 다수가 인지 |
| 대응 여유 | 시간 있음 | 즉시 대응 필요 |
| 경쟁 우위 | 선점 가능 | 모두가 대응 |
| 탐지 난이도 | 어려움 | 쉬움 |
| 예시 | 2010년 딥러닝 논문 | 2023년 ChatGPT 열풍 |

**약신호가 중요한 이유**: 약신호 단계에서 변화를 감지하면 **선점 기회**를 얻는다. 강신호가 되면 모두가 같은 정보를 갖게 되어 경쟁 우위가 사라진다.

### AI가 약신호 탐지에 강한 이유

1. **대량 정보 처리**: 수만 건의 뉴스, 논문, 특허를 자동 수집·분류
2. **이상 패턴 탐지**: 특정 키워드의 급격한 언급 증가, 새 단어 출현
3. **편향 없는 처리**: 확증 편향 없이 모든 정보를 동등하게 분석

In [ ]:
# 시뮬레이션: 약신호 탐지 — 키워드 트렌드 분석
# 배터리 산업 뉴스 키워드 트렌드 데이터 생성

days = 180
dates = pd.date_range(end=datetime.now(), periods=days, freq='D')

keywords = {
    'Lithium-ion': {'base': 15, 'trend': 'stable', 'noise': 3},
    'Solid-state': {'base': 8, 'trend': 'rising', 'noise': 2},
    'LFP': {'base': 10, 'trend': 'rising', 'noise': 3},
    'Sodium-ion': {'base': 3, 'trend': 'emerging', 'noise': 1.5},
    'Recycling': {'base': 7, 'trend': 'rising', 'noise': 2},
    'Supply Chain': {'base': 12, 'trend': 'volatile', 'noise': 5},
}

news_data = []
for date_idx, date in enumerate(dates):
    for kw, config in keywords.items():
        t = date_idx / days
        if config['trend'] == 'stable':
            value = config['base']
        elif config['trend'] == 'rising':
            value = config['base'] * (1 + 0.8 * t)
        elif config['trend'] == 'emerging':
            # 약신호: 초반 낮다가 후반 급등
            value = config['base'] * (1 + 3.0 * max(0, t - 0.5)**2)
        elif config['trend'] == 'volatile':
            value = config['base'] * (1 + 0.3 * np.sin(8 * np.pi * t))
        
        value += np.random.normal(0, config['noise'])
        value = max(0, value)
        news_data.append({'date': date, 'keyword': kw, 'mentions': value})

news_df = pd.DataFrame(news_data)

# 주별 집계
news_df['week'] = news_df['date'].dt.isocalendar().week.astype(int) + \
                  (news_df['date'].dt.year - news_df['date'].dt.year.min()) * 52
weekly = news_df.groupby(['week', 'keyword'])['mentions'].sum().reset_index()

# 시각화
fig = go.Figure()
colors = {'Lithium-ion': '#636EFA', 'Solid-state': '#EF553B', 'LFP': '#00CC96',
          'Sodium-ion': '#AB63FA', 'Recycling': '#FFA15A', 'Supply Chain': '#19D3F3'}

for kw in keywords:
    kw_data = weekly[weekly['keyword'] == kw]
    fig.add_trace(go.Scatter(
        x=kw_data['week'], y=kw_data['mentions'],
        mode='lines', name=kw, line=dict(color=colors[kw], width=2)
    ))

fig.update_layout(
    title='Battery Industry: Keyword Mention Trends (6 months)',
    xaxis_title='Week', yaxis_title='Weekly Mentions',
    height=450, hovermode='x unified'
)
fig.show()

print("💡 트렌드 해석:")
print("   - Lithium-ion: 안정적 (성숙 기술)")
print("   - Solid-state: 상승 (차세대 기술 주목)")
print("   - Sodium-ion: 후반 급등 → 약신호! (저비용 대안 부상)")
print("   - Supply Chain: 변동성 큼 (리스크 이슈)")

In [ ]:
# 약신호 탐지: 이상 탐지(Anomaly Detection) 알고리즘

def detect_weak_signals(weekly_df, threshold=2.0, window=4):
    """약신호 탐지: 이동평균 대비 급격한 증가 패턴 감지
    
    Args:
        weekly_df: 주별 키워드 언급량 데이터
        threshold: 이동평균 대비 비율 임계치
        window: 이동평균 윈도우 (주)
    Returns:
        약신호로 판별된 키워드 리스트
    """
    signals = []
    
    for kw in weekly_df['keyword'].unique():
        kw_data = weekly_df[weekly_df['keyword'] == kw].sort_values('week')
        mentions = kw_data['mentions'].values
        
        if len(mentions) < window + 1:
            continue
        
        # 이동평균 계산
        ma = pd.Series(mentions).rolling(window).mean().values
        
        # 최근 주의 비율
        recent = mentions[-1]
        ma_prev = ma[-2]  # 직전 이동평균
        
        if ma_prev > 0:
            ratio = recent / ma_prev
        else:
            ratio = 0
        
        # 성장률 계산 (초기 vs 최근)
        first_quarter = mentions[:len(mentions)//4].mean()
        last_quarter = mentions[-len(mentions)//4:].mean()
        
        if first_quarter > 0:
            growth_rate = (last_quarter - first_quarter) / first_quarter * 100
        else:
            growth_rate = 0
        
        signals.append({
            'keyword': kw,
            'recent_mentions': recent,
            'ma_ratio': ratio,
            'growth_rate': growth_rate,
            'is_weak_signal': ratio > threshold or growth_rate > 100
        })
    
    return pd.DataFrame(signals)

# 약신호 탐지 실행
signal_df = detect_weak_signals(weekly)

# 결과 시각화
fig = px.scatter(signal_df, x='growth_rate', y='ma_ratio',
                 text='keyword', size='recent_mentions',
                 color='is_weak_signal',
                 color_discrete_map={True: '#e74c3c', False: '#3498db'},
                 labels={'growth_rate': 'Growth Rate (%)', 
                         'ma_ratio': 'Recent / Moving Avg Ratio',
                         'is_weak_signal': 'Weak Signal?'})

fig.add_hline(y=2.0, line_dash='dash', line_color='orange',
              annotation_text='Anomaly Threshold')
fig.add_vline(x=100, line_dash='dash', line_color='orange',
              annotation_text='High Growth Threshold')

fig.update_traces(textposition='top center')
fig.update_layout(
    title='Weak Signal Detection: Growth Rate vs Anomaly Ratio',
    height=450
)
fig.show()

print("\n📊 약신호 탐지 결과:")
for _, row in signal_df.iterrows():
    status = "🚨 약신호!" if row['is_weak_signal'] else "✅ 정상"
    print(f"  {row['keyword']}: 성장률 {row['growth_rate']:.0f}%, "
          f"비율 {row['ma_ratio']:.2f} → {status}")

### 7.2.3 감성 분석 (Sentiment Analysis)

소셜 미디어와 뉴스의 감정적 톤을 분류하여 여론의 흐름을 파악한다.

| 접근법 | 대표 도구 | 특징 |
|--------|----------|------|
| 사전 기반 | VADER | 규칙 기반, 해석 용이 |
| 기계학습 | SVM, BERT | 도메인 특화 가능 |
| 사전학습 모델 | GPT, BERT | 문맥 이해, 높은 정확도 |

**활용 사례:**
- 브랜드 모니터링: 긍정 비율 60% → 40%로 하락 시 위기 신호
- 제품 출시 반응: 초기 반응 긍정/부정 파악
- 경쟁사 평판: 경쟁사 제품 문제 부각 시 우리의 기회

In [ ]:
# 감성 분석 시뮬레이션: 키워드별 감성 점수 추이

# 시뮬레이션 감성 데이터 생성
sentiment_data = []
for date_idx, date in enumerate(dates):
    t = date_idx / days
    for kw, base_sentiment in {
        'Solid-state': 0.45, 'Sodium-ion': 0.52, 'Recycling': 0.31,
        'LFP': 0.22, 'Supply Chain': -0.28, 'Safety': -0.35,
        'Lithium-ion': 0.15, 'Energy Density': 0.28
    }.items():
        # 시간에 따른 변동
        noise = np.random.normal(0, 0.15)
        sentiment = base_sentiment + noise
        if kw == 'Supply Chain':
            sentiment += 0.2 * np.sin(4 * np.pi * t)  # 변동성
        sentiment = np.clip(sentiment, -1, 1)
        sentiment_data.append({'date': date, 'keyword': kw, 'sentiment': sentiment})

sent_df = pd.DataFrame(sentiment_data)
sent_df['month'] = sent_df['date'].dt.to_period('M').astype(str)

# 월별 평균 감성
monthly_sent = sent_df.groupby(['month', 'keyword'])['sentiment'].mean().reset_index()

# 히트맵 데이터
pivot = monthly_sent.pivot(index='keyword', columns='month', values='sentiment')

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=pivot.columns.tolist(),
    y=pivot.index.tolist(),
    colorscale='RdYlGn',
    zmin=-0.8, zmax=0.8,
    text=np.round(pivot.values, 2),
    texttemplate='%{text}',
    textfont=dict(size=10)
))

fig.update_layout(
    title='Keyword Sentiment Heatmap (Monthly Average)',
    xaxis_title='Month', yaxis_title='Keyword',
    height=400
)
fig.show()

print("💡 감성 분석 결과:")
print("   - Solid-state, Sodium-ion: 긍정적 → 기대감 높은 기술")
print("   - Supply Chain, Safety: 부정적 → 리스크/위험 이슈")
print("   - 감성 점수의 급변은 중요 이벤트 발생 신호!")

### 7.2.4 LLM을 활용한 통합 분석

대규모 언어모델(LLM)은 트렌드 분석을 한 단계 발전시킨다:

| 활용 | 설명 | 예시 |
|------|------|------|
| 대량 문서 요약 | 수백 개 기사를 핵심 요약 | "이번 달 배터리 산업 주요 이슈는?" |
| 주요 이슈 추출 | 비구조화 텍스트에서 구조화된 정보 | 기업명, 기술명, 날짜 추출 |
| 인사이트 생성 | 트렌드의 전략적 함의 초안 | "이 트렌드가 우리 사업에 미치는 영향" |
| 비교 분석 | 복수 소스 종합 비교 | "경쟁사 A와 B의 전략 차이" |

⚠️ **주의: LLM 환각(Hallucination)** — 존재하지 않는 사실을 생성할 수 있다.

**환각 통제 방법:**
1. 출처 명시 요구 ("인용한 출처와 날짜를 명시하라")
2. 교차검증 수행 (최소 2개 독립 출처 확인)
3. RAG 기반 검색 제한 (검색된 문서만 맥락으로 제공)
4. 확신도 표시 요청 ("불확실한 부분을 명시하라")

### 📝 이론 과제 7-1-2 (15분)

#### 과제: 약신호와 감성 분석

**질문:**

1. **약신호가 강신호로 전환된 역사적 사례**를 하나 검색하시오. 약신호 단계에서 어떤 특징이 있었고, 어떤 기업이 선점에 성공/실패했는지 설명하시오. (4-5문장)

2. **감성 분석(Sentiment Analysis)**이 브랜드 위기관리에 활용된 실제 사례를 검색하시오. 어떤 도구를 사용했고, 어떤 조기 경보를 감지했는지 설명하시오. (3-4문장)

3. **LLM의 환각(Hallucination)** 문제가 환경분석에서 왜 특히 위험한지 설명하시오. RAG(Retrieval-Augmented Generation)가 이를 어떻게 완화하는지 검색하시오. (3-4문장)

**조사 키워드:**
- "weak signal to strong signal example"
- "sentiment analysis brand crisis management"
- "RAG retrieval augmented generation hallucination"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-1-2 제출란

**학번:** ___________
**이름:** ___________

#### 1. 약신호 → 강신호 전환 사례

(여기에 작성)

#### 2. 감성 분석의 브랜드 위기관리 활용 사례

(여기에 작성)

#### 3. LLM 환각의 위험성과 RAG

(여기에 작성)

**출처:** (URL)

---

## ☕ 휴식 (15분)

---

---

## Part 3. 경쟁 인텔리전스

---

### 7.3 경쟁 인텔리전스 (Competitive Intelligence)

경쟁 인텔리전스(CI)는 경쟁 환경에 대한 정보를 체계적으로 수집·분석하여 의사결정에 활용하는 활동이다.

### 7.3.1 자동 추적 가능한 공개 정보원

| 정보원 | 획득 정보 | 분석 방법 | 접근성 |
|--------|----------|----------|--------|
| 뉴스/보도자료 | 전략 발표, 실적 | 키워드 모니터링 | 높음 |
| 채용 공고 | 투자 방향, 역량 | 직무 분류 분석 | 높음 |
| 특허 DB | 기술 전략, R&D | 출원 동향 분석 | 높음 |
| 논문/학회 | 연구 방향 | 저자/기관 분석 | 높음 |
| 공시/IR | 재무, 전략 | 텍스트 마이닝 | 중간 |
| 가격/제품 정보 | 마케팅 전략 | 시계열 추적 | 높음 |

### 채용 공고 = 전략의 선행 지표

채용 공고는 기업 전략의 **선행 지표(Leading Indicator)**이다:
- AI 엔지니어 대거 채용 → AI 투자 예고
- 해외 사무소 채용 증가 → 해외 진출 암시
- 특정 분야 전문가 채용 → 해당 분야 진출 신호
- 채용 동결, 시니어 이직 증가 → 구조조정 신호

In [ ]:
# 경쟁 인텔리전스: 특허 동향 분석 시뮬레이션

companies = ["CATL", "LG Energy", "Samsung SDI", "BYD", "SK On", "Panasonic"]
tech_areas = ["Solid-state", "Li-ion Cathode", "Si Anode", "BMS", "Recycling", "Sodium-ion"]

# 특허 데이터 시뮬레이션
patent_data = []
current_year = 2026

tech_config = {
    'Solid-state': {'trend': 'rising', 'leaders': ['Samsung SDI', 'Panasonic']},
    'Li-ion Cathode': {'trend': 'stable', 'leaders': ['CATL', 'LG Energy']},
    'Si Anode': {'trend': 'rising', 'leaders': ['Panasonic', 'Samsung SDI']},
    'BMS': {'trend': 'stable', 'leaders': ['CATL', 'BYD']},
    'Recycling': {'trend': 'emerging', 'leaders': ['LG Energy', 'SK On']},
    'Sodium-ion': {'trend': 'emerging', 'leaders': ['CATL', 'BYD']},
}

for year in range(current_year - 5, current_year + 1):
    for company in companies:
        for tech, config in tech_config.items():
            base = 12 if company in config['leaders'] else 5
            year_idx = year - (current_year - 5)
            
            if config['trend'] == 'rising':
                multiplier = 1 + year_idx * 0.15
            elif config['trend'] == 'emerging':
                multiplier = 0.3 + year_idx * 0.25 if year_idx > 2 else 0.3
            else:
                multiplier = 1.0
            
            count = int(base * multiplier * np.random.uniform(0.7, 1.3))
            patent_data.append({
                'year': year, 'company': company,
                'tech_area': tech, 'patents': count
            })

patent_df = pd.DataFrame(patent_data)

# 시각화 1: 기업별 총 특허 출원 추이
yearly_total = patent_df.groupby(['year', 'company'])['patents'].sum().reset_index()

fig = px.line(yearly_total, x='year', y='patents', color='company',
              title='Patent Filing Trends by Company',
              labels={'patents': 'Total Patents', 'year': 'Year'})
fig.update_layout(height=400)
fig.show()

# 시각화 2: 기술 분야별 최근 성장률
recent = patent_df[patent_df['year'] >= current_year - 2].groupby('tech_area')['patents'].sum()
early = patent_df[patent_df['year'] <= current_year - 4].groupby('tech_area')['patents'].sum()
growth = ((recent - early) / early * 100).sort_values(ascending=True)

fig2 = go.Figure(go.Bar(
    x=growth.values, y=growth.index,
    orientation='h',
    marker_color=['#e74c3c' if v > 50 else '#3498db' for v in growth.values]
))
fig2.update_layout(
    title='Patent Growth Rate by Technology Area (Recent 2yr vs Early 2yr)',
    xaxis_title='Growth Rate (%)', height=350
)
fig2.show()

print("💡 특허 분석 시사점:")
print("   - 전고체(Solid-state): 삼성SDI, 파나소닉 주도 — 기술 선점 경쟁")
print("   - 나트륨이온(Sodium-ion): 급부상 — CATL, BYD 선점")
print("   - 재활용(Recycling): 신흥 분야 — 규제 연계 성장")

In [ ]:
# 경쟁 인텔리전스: 채용 공고 분석

# 채용 데이터 시뮬레이션
job_categories = ['R&D', 'Manufacturing', 'Sales/Marketing', 'Software/AI', 'Supply Chain']

hiring_data = []
for company in companies:
    total = np.random.randint(200, 800)
    
    # 기업별 채용 비중 (전략 반영)
    if company == 'CATL':
        weights = [0.2, 0.35, 0.15, 0.15, 0.15]  # 생산 중심
    elif company == 'Samsung SDI':
        weights = [0.35, 0.2, 0.1, 0.25, 0.1]  # R&D 중심
    elif company == 'BYD':
        weights = [0.15, 0.3, 0.2, 0.1, 0.25]  # 수직계열화
    elif company == 'SK On':
        weights = [0.2, 0.3, 0.15, 0.15, 0.2]  # 확장 중심
    else:
        weights = [0.25, 0.25, 0.15, 0.2, 0.15]
    
    for cat, w in zip(job_categories, weights):
        hiring_data.append({
            'company': company, 'category': cat,
            'postings': int(total * w)
        })

hiring_df = pd.DataFrame(hiring_data)

# 시각화: 기업별 채용 구성
fig = px.bar(hiring_df, x='company', y='postings', color='category',
             barmode='stack',
             title='Hiring Profile by Company (Job Category Distribution)',
             labels={'postings': 'Job Postings', 'category': 'Category'})
fig.update_layout(height=400)
fig.show()

# 전략 추론
print("💡 채용 분석을 통한 전략 추론:")
for company in companies:
    comp_data = hiring_df[hiring_df['company'] == company]
    top_cat = comp_data.loc[comp_data['postings'].idxmax(), 'category']
    total = comp_data['postings'].sum()
    sw_ratio = comp_data[comp_data['category'] == 'Software/AI']['postings'].values[0] / total * 100
    print(f"  {company}: 최다 채용 = {top_cat}, AI/SW 비중 = {sw_ratio:.0f}%")

### 7.3.3 경쟁 인텔리전스 윤리

경쟁 인텔리전스와 산업 스파이는 다르다.

**합법적 정보 수집 원칙:**
- 공개된 정보만 활용 (뉴스, 공시, 특허, 웹사이트)
- 허위 신분으로 정보 획득 금지
- 영업비밀 침해 금지
- robots.txt 존중, 과도한 크롤링 금지
- GDPR, 개인정보보호법 준수

---

### 📝 이론 과제 7-1-3 (10분)

#### 과제: 경쟁 인텔리전스 실습

**질문:**

1. **채용 공고가 전략의 선행 지표**인 실제 사례를 검색하시오. 어떤 기업의 채용 변화가 어떤 전략 변화를 예고했는가? (3-4문장)

2. **특허 분석**으로 경쟁사 전략을 파악한 사례를 검색하시오. 어떤 지표(출원 건수, 기술 분야, 피인용)를 사용했는가? (3-4문장)

**조사 키워드:**
- "hiring trends predict company strategy"
- "patent analysis competitive intelligence"

**제출:** 아래 마크다운 셀에 **직접 타이핑** (복사 붙여넣기 금지)

### ✅ 과제 7-1-3 제출란

**학번:** ___________
**이름:** ___________

#### 1. 채용 공고 → 전략 예측 사례

(여기에 작성)

#### 2. 특허 분석 → 경쟁사 전략 파악 사례

(여기에 작성)

**출처:** (URL)

---

## Part 4. 실습: 뉴스 트렌드 분석 & 약신호 탐지

---

### 실습 7-1-4: 트렌드 분석 함수 구현

앞서 시뮬레이션한 뉴스 데이터를 활용하여 트렌드 분석과 약신호 탐지를 직접 구현합니다.

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# TODO 1: 키워드별 6개월 성장률 계산 함수
# 힌트: 초기 4주 평균 vs 최근 4주 평균 비교
def calculate_growth_rate(weekly_df, keyword):
    """키워드별 성장률 계산
    
    Args:
        weekly_df: 주별 키워드 데이터
        keyword: 분석할 키워드
    Returns:
        float: 성장률 (%)
    """
    kw_data = weekly_df[weekly_df['keyword'] == keyword].sort_values('week')
    mentions = kw_data['mentions'].values
    
    # TODO: 초기 4주 평균과 최근 4주 평균을 비교하여 성장률 계산
    first_period = None  # 여기에 코드 작성
    last_period = None   # 여기에 코드 작성
    growth = None        # 여기에 코드 작성
    
    return growth

# TODO 2: 모든 키워드에 대해 성장률 계산 후 DataFrame으로 정리
# 힌트: for kw in keywords: 루프 활용

# TODO 3: 성장률 기준으로 정렬하여 막대 차트 시각화
# 힌트: px.bar() 사용, 색상으로 성장률 크기 구분

# TODO 4: 결과 해석 (print문으로)
# - 가장 빠르게 성장하는 키워드는?
# - 약신호로 판단되는 키워드는?
# - 비즈니스 함의는?

# ========== 여기까지 ==========

print("\n💡 해석 가이드:")
print("   성장률 > 100%: 급성장 (약신호 가능성)")
print("   성장률 > 50%: 상승 트렌드")
print("   성장률 0~50%: 안정적")
print("   성장률 < 0%: 하락 트렌드")

### 실습 7-1-5: 감성 분석 시각화

아래 TODO를 완성하여 키워드별 감성 점수를 분석하고, **성장률 × 감성** 매트릭스를 만드시오.

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# 감성 데이터 월별 평균 (이미 계산됨: monthly_sent)
# 키워드별 평균 감성 점수 계산
avg_sentiment = sent_df.groupby('keyword')['sentiment'].mean().reset_index()
avg_sentiment.columns = ['keyword', 'avg_sentiment']

# TODO 1: signal_df와 avg_sentiment를 merge하여 통합 DataFrame 생성
# 힌트: pd.merge(signal_df, avg_sentiment, on='keyword')

# TODO 2: 성장률(x축) × 감성점수(y축) 산점도 작성
# 힌트: go.Figure() + go.Scatter(), 버블 크기 = 언급량
# 4개 사분면을 해석:
#   - 우상: 성장 + 긍정 → ⭐ 핵심 기회
#   - 우하: 성장 + 부정 → ⚠️ 리스크 이슈
#   - 좌상: 정체 + 긍정 → 안정 기술
#   - 좌하: 정체 + 부정 → 위험 요소

# TODO 3: 사분면 경계선 추가 (x=50, y=0)

# TODO 4: 결과 해석 (print문으로)
# - 핵심 기회: 어떤 키워드?
# - 주요 리스크: 어떤 키워드?
# - 전략적 시사점은?

# ========== 여기까지 ==========

---

## Part 5. 실습: 경쟁사 프로파일링 & 데이터 기반 SWOT

---

### 실습 7-1-6: 경쟁사 기술 포트폴리오 분석

특허 데이터를 활용하여 경쟁사별 기술 포트폴리오를 시각화하고 전략을 추론합니다.

In [ ]:
# 경쟁사 기술 포트폴리오 분석 (최근 2년)
recent_patents = patent_df[patent_df['year'] >= current_year - 1]
portfolio = recent_patents.groupby(['company', 'tech_area'])['patents'].sum().reset_index()

# 레이더 차트: 기업별 기술 포트폴리오
fig = go.Figure()

top_companies = ['CATL', 'Samsung SDI', 'LG Energy', 'BYD']
radar_colors = ['#636EFA', '#EF553B', '#00CC96', '#AB63FA']

for company, color in zip(top_companies, radar_colors):
    comp_data = portfolio[portfolio['company'] == company]
    values = []
    for tech in tech_areas:
        val = comp_data[comp_data['tech_area'] == tech]['patents'].values
        values.append(val[0] if len(val) > 0 else 0)
    
    fig.add_trace(go.Scatterpolar(
        r=values + [values[0]],  # close the polygon
        theta=tech_areas + [tech_areas[0]],
        fill='toself', name=company,
        line=dict(color=color)
    ))

fig.update_layout(
    polar=dict(radialaxis=dict(visible=True)),
    title='Technology Portfolio Comparison (Recent Patents)',
    height=500
)
fig.show()

print("💡 기술 포트폴리오 분석:")
print("   - CATL: 리튬이온/BMS 강점, 나트륨이온 선점 중")
print("   - Samsung SDI: 전고체/Si 음극재 집중 (기술 차별화 전략)")
print("   - LG Energy: 균형 잡힌 포트폴리오, 재활용 강화 중")
print("   - BYD: 수직계열화 반영, 공급망/BMS 강점")

### 📝 실습 과제 7-1-7 (15분)

#### 과제: 데이터 기반 SWOT 작성

배터리 소재 기업 C사 관점에서, 앞서 분석한 데이터를 근거로 **데이터 기반 SWOT**을 작성하시오.

In [ ]:
# ========== 여기서부터 코드를 작성하세요 ==========

# C사 배경: 정밀화학 소재 기업, 배터리 시장 진출 검토 중
# 강점: 소재 합성 기술 (특허 32건), 정밀화학 제조 역량, R&D 인력 (박사 45명)
# 약점: 배터리 산업 경험 없음, 브랜드 인지도 없음, 규모 열위

# TODO 1: SWOT 매트릭스 데이터 정의
# 힌트: 각 항목에 "데이터 근거"를 반드시 포함
swot_data = {
    'Strengths': [
        # ('항목', '데이터 근거')
    ],
    'Weaknesses': [
        # ('항목', '데이터 근거')
    ],
    'Opportunities': [
        # ('항목', '데이터 근거') — 앞서 분석한 트렌드/감성 결과 활용
    ],
    'Threats': [
        # ('항목', '데이터 근거') — 앞서 분석한 경쟁사/리스크 결과 활용
    ]
}

# TODO 2: SWOT을 Plotly Table로 시각화
# 힌트: go.Figure(data=[go.Table(...)]) 사용

# TODO 3: 기회/위협 정량화 매트릭스 (영향도 × 확률)
# 힌트: go.Scatter()로 버블 차트, x=확률, y=영향도, size=우선순위

# TODO 4: 전략적 시사점 도출 (print문으로)
# - 타겟 시장은?
# - 진출 방식은? (자체 진출 vs JV vs 인수)
# - 핵심 차별화 포인트는?

# ========== 여기까지 ==========

---

## 🎓 강의 마무리 및 핵심 요약

### 오늘 배운 내용

1. **전통적 프레임워크의 한계**: PEST, SWOT, 5 Forces는 여전히 유용하지만, 정적·주관적·정보처리 한계가 있다. 변화의 속도가 분석의 속도를 앞지르는 시대에 새로운 접근이 필요하다

2. **AI 기반 환경 스캐닝**: 실시간 뉴스 수집, 감성 분석, 이상 탐지, LLM 기반 요약으로 정적 분석에서 동적 모니터링으로 전환할 수 있다

3. **약신호 포착이 경쟁 우위의 원천**: 미래 변화의 초기 징후를 선점하면 경쟁 우위를 얻는다. AI는 대량 정보에서 이상 패턴을 탐지하여 약신호를 식별하는 데 강점이 있다

4. **경쟁 인텔리전스 자동화**: 뉴스, 특허, 채용 공고 등 공개 데이터를 체계적으로 분석하면 경쟁사의 전략 방향을 추론할 수 있다. 합법적·윤리적 범위 내에서 자동화 가능하다

5. **데이터 기반 SWOT**: "강점"이라고 주장하려면 데이터로 뒷받침해야 한다. 기회/위협 정량화로 우선순위를 정할 수 있다

### 핵심 메시지

> 환경은 더 이상 연간 보고서로 분석할 수 없다. **AI를 활용한 실시간 환경 스캐닝 역량**이 전략적 민첩성의 기반이다. 그러나 AI는 도구일 뿐, **최종 판단과 전략 수립은 여전히 인간의 몫**이다. Human-AI 협업이 핵심이다.

### 다음 장 예고

**제7-2장: 이해관계자와 네트워크 분석** — 환경 분석에서 "누가 관련되어 있는가"를 파악한다. 이해관계자 맵핑, 권력-이해 매트릭스, 사회적 네트워크 분석(SNA)을 학습한다.

### 📚 추가 학습 자료

- Ansoff, H.I. (1975). Managing Strategic Surprise by Response to Weak Signals. *California Management Review*, 18(2), 21-33.
- Porter, M.E. (1985). *Competitive Advantage*. Free Press.
- Day, G.S. & Schoemaker, P.J. (2019). *See Sooner, Act Faster*. MIT Press.
- Fleisher, C.S. & Bensoussan, B.E. (2015). *Business and Competitive Analysis*. FT Press.
- **Python 도구**: NewsAPI, VADER (감성분석), BERTopic (토픽 모델링)
- **특허 검색**: Google Patents, USPTO, KIPRIS

---

**제7-1장 끝**